# 03 — Enums e guardrails

**Trilha:** [01 chamada e JSON](01_chamada_e_json.ipynb) → [02 saída estruturada](02_saida_estruturada.ipynb) → **03 enums e guardrails** → [04 ferramentas com funções](../02_intermediario/04_ferramentas_com_funcoes.ipynb)

**De onde viemos:** no [02](02_saida_estruturada.ipynb), o contrato garantiu a forma, mas `unidade: str` aceitou `"cups"` e
`"tablespoon"`. Aqui o vocabulário de cada campo vira um **enum**: só os valores listados passam. E o mesmo recurso vira um
**guardrail**: o modelo precisa dizer se o pedido está no escopo, e o código decide o que fazer com cada caso.

Os enums criados aqui (`Unidade`, `Categoria`, `TipoPedido`) acompanham o resto da trilha.

## 1. Enums para o vocabulário

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # saida.py e logger_general.py ficam em AI_use_cases/
from saida import mostrar, mostrar_prompt, tabela

In [2]:
import json
from enum import StrEnum
from typing import Annotated

from pydantic import BaseModel, BeforeValidator, ConfigDict, Field, ValidationError, model_validator


class Unidade(StrEnum):
    G = "g"
    KG = "kg"
    ML = "ml"
    L = "l"
    UN = "un"


class Categoria(StrEnum):
    HORTIFRUTI = "hortifruti"
    LATICINIOS = "laticinios"
    CARNES = "carnes"
    MERCEARIA = "mercearia"
    PADARIA = "padaria"


mostrar(f"list(Unidade): {[u.value for u in Unidade]}")
mostrar(f"Unidade('g'): {Unidade('g')!r}")
mostrar(f"Unidade.G == 'g': {Unidade.G == 'g'}")
mostrar(f"f'{{Unidade.KG}}': {Unidade.KG}")

list(Unidade): ['g', 'kg', 'ml', 'l', 'un']
Unidade('g'): <Unidade.G: 'g'>
Unidade.G == 'g': True
f'{Unidade.KG}': kg


`StrEnum` é um enum cujos membros **são** strings: comparam igual ao texto e aparecem como texto em f-strings e no JSON.

In [3]:
class Ingrediente(BaseModel):
    model_config = ConfigDict(extra="forbid")
    nome: str
    quantidade: float = Field(gt=0)
    unidade: Unidade
    categoria: Categoria


class ListaDeCompras(BaseModel):
    model_config = ConfigDict(extra="forbid")
    prato: str
    porcoes: int = Field(gt=0)
    ingredientes: list[Ingrediente]


schema = ListaDeCompras.model_json_schema()
mostrar(f"Unidade no schema: {schema['$defs']['Unidade']}")
mostrar(f"Categoria no schema: {schema['$defs']['Categoria']['enum']}")

Unidade no schema: {'enum': ['g', 'kg', 'ml', 'l', 'un'], 'title': 'Unidade', 'type': 'string'}
Categoria no schema: ['hortifruti', 'laticinios', 'carnes', 'mercearia', 'padaria']


Diferente do `description` do 02, o `enum` no schema é uma **lista fechada**: o modelo vê as opções, e o Pydantic recusa
qualquer outra. O caso `"cups"` agora falha (*exemplo construído*):

In [4]:
imperial = {"nome": "flour", "quantidade": 2, "unidade": "cups", "categoria": "mercearia"}
try:
    Ingrediente.model_validate(imperial)
except ValidationError as ex:
    erro = ex.errors()[0]
    mostrar(f"{erro['loc']}: {erro['msg']}")

('unidade',): Input should be 'g', 'kg', 'ml', 'l' or 'un'


Ganho prático do enum: organizar a lista **por seção do mercado**, na ordem em que as categorias foram declaradas
(*exemplo construído*):

In [5]:
resposta = """
{"prato": "Pizza margherita", "porcoes": 2, "ingredientes": [
  {"nome": "farinha de trigo", "quantidade": 250, "unidade": "g", "categoria": "mercearia"},
  {"nome": "muçarela de búfala", "quantidade": 150, "unidade": "g", "categoria": "laticinios"},
  {"nome": "manjericão fresco", "quantidade": 1, "unidade": "un", "categoria": "hortifruti"},
  {"nome": "tomate italiano", "quantidade": 4, "unidade": "un", "categoria": "hortifruti"},
  {"nome": "azeite", "quantidade": 15, "unidade": "ml", "categoria": "mercearia"}
]}
"""
lista = ListaDeCompras.model_validate_json(resposta)

secoes = []
for categoria in Categoria:
    itens = [item for item in lista.ingredientes if item.categoria is categoria]
    if itens:
        secoes.append((categoria, ", ".join(f"{item.nome} ({item.quantidade:g} {item.unidade})" for item in itens)))
tabela(["seção", "itens"], secoes, titulo="lista por seção do mercado")

lista por seção do mercado                                       
                                                                 
  seção        itens                                             
 ─────────────────────────────────────────────────────────────── 
  hortifruti   manjericão fresco (1 un), tomate italiano (4 un)  
  laticinios   muçarela de búfala (150 g)                        
  mercearia    farinha de trigo (250 g), azeite (15 ml)          
                                                                 


## 2. Guardrail: o pedido está no escopo?

Um assistente de lista de compras recebe de tudo. Em vez de torcer para o modelo "se comportar", o contrato **obriga** a resposta
a declarar o tipo — e o código trata cada tipo explicitamente.

In [6]:
class TipoPedido(StrEnum):
    RECEITA = "receita"
    FORA_DO_ESCOPO = "fora_do_escopo"


class RespostaAssistente(BaseModel):
    model_config = ConfigDict(extra="forbid")
    tipo: TipoPedido
    lista: ListaDeCompras | None = None
    motivo: str | None = Field(default=None, description="Obrigatório quando tipo = fora_do_escopo")

    @model_validator(mode="after")
    def tipo_combina_com_conteudo(self):
        if self.tipo is TipoPedido.RECEITA and self.lista is None:
            raise ValueError("tipo 'receita' exige 'lista'")
        if self.tipo is TipoPedido.FORA_DO_ESCOPO and (self.lista is not None or not self.motivo):
            raise ValueError("tipo 'fora_do_escopo' exige 'motivo' e não aceita 'lista'")
        return self


def tratar(resposta: RespostaAssistente) -> str:
    match resposta.tipo:
        case TipoPedido.RECEITA:
            return f"Lista de {resposta.lista.prato}: {len(resposta.lista.ingredientes)} itens"
        case TipoPedido.FORA_DO_ESCOPO:
            return f"Só ajudo com listas de compras. ({resposta.motivo})"

O `model_validator` impede combinações sem sentido (tipo "receita" sem lista, por exemplo). Com isso, `tratar` pode confiar que
`resposta.lista` existe no caso `RECEITA`.

In [ ]:
# api: configuração do cliente — lê a chave e o endpoint do .env
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("openai")
if not api_key:
    raise ValueError("variável 'openai' (chave da API) não encontrada no .env")
openai_base_url = os.getenv("openai_base_url")
if not openai_base_url:
    raise ValueError("variável 'openai_base_url' não encontrada no .env")

AI_MODEL = "gpt-4o-mini"
ai_client = OpenAI(api_key=api_key, base_url=openai_base_url)

In [7]:
def prompt_assistente() -> str:
    return (
        "Você é um assistente de listas de compras. Se o usuário pedir um prato ou receita, gere a lista com ingredientes "
        "in natura e sistema métrico (tipo = receita). Para qualquer outro pedido, responda tipo = fora_do_escopo "
        "com um motivo curto.\n"
        "Responda SOMENTE com um JSON válido que siga este JSON Schema:\n"
        f"{json.dumps(RespostaAssistente.model_json_schema(), ensure_ascii=False)}"
    )


def extrair_json(texto: str) -> str:
    inicio, fim = texto.find("{"), texto.rfind("}")
    if inicio == -1 or fim == -1:
        raise ValueError("nenhum objeto JSON na resposta")
    return texto[inicio:fim + 1]


mostrar_prompt("prompt_assistente()", prompt_assistente(), compacto=True)

╭─ prompt_assistente() ──────────────────────────────────────────────────────────────────────────────────────╮
│ Você é um assistente de listas de compras. Se o usuário pedir um prato ou receita, gere a lista com        │
│ ingredientes in natura e sistema métrico (tipo = receita). Para qualquer outro pedido, responda tipo =     │
│ fora_do_escopo com um motivo curto.                                                                        │
│ Responda SOMENTE com um JSON válido que siga este JSON Schema:                                             │
│ {"$defs": {"Categoria": {"enum": ["hortifruti", "laticinios", "carnes", "mercearia", "padaria"], "title":  │
│ "Categoria", "type": "string"}, "Ingrediente": {"additionalProperties": false, "properties": {"nome":      │
│ {"title": "Nome", "type": "string"}, "quantidade": {"exclusiveMinimum": 0, "title": "Quantidade", "type":  │
│ "number"}, "unidade": {"$ref": "#/$defs/Unidade"}, "categoria": {"$ref": "#/$defs/Categoria"}},            │
│

In [ ]:
# api: dois pedidos — um no escopo, outro fora
for pedido in ["Pizza margherita para 2 pessoas", "Qual a capital da França?"]:
    chat_completion = ai_client.chat.completions.create(
        model=AI_MODEL,
        messages=[{"role": "system", "content": prompt_assistente()}, {"role": "user", "content": pedido}],
    )
    texto = chat_completion.choices[0].message.content
    resposta_validada = RespostaAssistente.model_validate_json(extrair_json(texto))
    mostrar(f"{pedido!r} -> {tratar(resposta_validada)}")

Sem chave, o mesmo fluxo com respostas no formato esperado (*exemplos construídos*):

In [8]:
respostas_construidas = {
    "Pizza margherita para 2 pessoas": '{"tipo": "receita", "lista": ' + resposta.strip() + "}",
    "Qual a capital da França?": '{"tipo": "fora_do_escopo", "motivo": "pergunta de geografia, não de compras"}',
}
for pedido, texto in respostas_construidas.items():
    resposta_validada = RespostaAssistente.model_validate_json(extrair_json(texto))
    mostrar(f"{pedido!r} -> {tratar(resposta_validada)}")

'Pizza margherita para 2 pessoas' -> Lista de Pizza margherita: 5 itens
'Qual a capital da França?' -> Só ajudo com listas de compras. (pergunta de geografia, não de compras)


E as combinações incoerentes são barradas antes de chegar ao `tratar`:

In [9]:
incoerentes = {
    "receita sem lista": '{"tipo": "receita"}',
    "fora do escopo sem motivo": '{"tipo": "fora_do_escopo"}',
    "tipo inventado": '{"tipo": "piada", "motivo": "o usuário pediu uma piada"}',
}
for nome, texto in incoerentes.items():
    try:
        RespostaAssistente.model_validate_json(texto)
    except ValidationError as ex:
        mostrar(f"{nome}: {ex.errors()[0]['msg']}")

receita sem lista: Value error, tipo 'receita' exige 'lista'
fora do escopo sem motivo: Value error, tipo 'fora_do_escopo' exige 'motivo' e não aceita 'lista'
tipo inventado: Input should be 'receita' or 'fora_do_escopo'


## ⚠️ O que ainda quebra

**1. Enum compara texto exato.** `"G"` ou `" g"` não são `"g"` — e modelos variam caixa e espaços:

In [10]:
for valor in ["g", "G", " g", "gramas"]:
    try:
        mostrar(f"{valor!r}: {Unidade(valor)!r}")
    except ValueError as ex:
        mostrar(f"{valor!r}: ValueError: {ex}")

'g': <Unidade.G: 'g'>
'G': ValueError: 'G' is not a valid Unidade
' g': ValueError: ' g' is not a valid Unidade
'gramas': ValueError: 'gramas' is not a valid Unidade


Para variações **inofensivas** (caixa, espaços), normalizar antes de validar resolve. Sinônimos (`"gramas"`) é decisão de
negócio: ou entram numa tabela explícita, ou continuam sendo erro.

In [11]:
def normalizar_texto(valor):
    return valor.strip().lower() if isinstance(valor, str) else valor


UnidadeTolerante = Annotated[Unidade, BeforeValidator(normalizar_texto)]


class IngredienteTolerante(Ingrediente):
    unidade: UnidadeTolerante


item = IngredienteTolerante.model_validate({"nome": "sal", "quantidade": 5, "unidade": " G", "categoria": "mercearia"})
mostrar(f"item.unidade: {item.unidade!r}")

item.unidade: <Unidade.G: 'g'>


**2. `match` sem `case _` falha em silêncio quando o enum cresce.** Se amanhã entrar um tipo novo e ninguém atualizar o `match`,
a função devolve `None` sem erro:

In [12]:
class TipoPedidoV2(StrEnum):
    RECEITA = "receita"
    FORA_DO_ESCOPO = "fora_do_escopo"
    SUBSTITUICAO = "substituicao"  # novo


def rotular(tipo: TipoPedidoV2) -> str:
    match tipo:
        case TipoPedidoV2.RECEITA:
            return "gerar lista"
        case TipoPedidoV2.FORA_DO_ESCOPO:
            return "recusar"


mostrar(f"rotular(SUBSTITUICAO): {rotular(TipoPedidoV2.SUBSTITUICAO)!r}")


def rotular_seguro(tipo: TipoPedidoV2) -> str:
    match tipo:
        case TipoPedidoV2.RECEITA:
            return "gerar lista"
        case TipoPedidoV2.FORA_DO_ESCOPO:
            return "recusar"
        case _:
            raise ValueError(f"tipo sem tratamento: {tipo!r}")


try:
    rotular_seguro(TipoPedidoV2.SUBSTITUICAO)
except ValueError as ex:
    mostrar(f"rotular_seguro(SUBSTITUICAO): ValueError: {ex}")

rotular(SUBSTITUICAO): None
rotular_seguro(SUBSTITUICAO): ValueError: tipo sem tratamento: <TipoPedidoV2.SUBSTITUICAO: 'substituicao'>


**3. Enum garante o vocabulário, não o bom senso.** Tudo abaixo é válido pelo contrato (*exemplo construído*):

In [13]:
absurda = {
    "prato": "Pizza margherita", "porcoes": 2, "ingredientes": [
        {"nome": "sal", "quantidade": 3, "unidade": "kg", "categoria": "mercearia"},
        {"nome": "muçarela de búfala", "quantidade": 40, "unidade": "un", "categoria": "laticinios"},
        {"nome": "farinha de trigo", "quantidade": 250, "unidade": "g", "categoria": "laticinios"},
    ],
}
lista_absurda = ListaDeCompras.model_validate(absurda)
tabela(["ingrediente", "quantidade", "categoria"],
       [(item.nome, f"{item.quantidade:g} {item.unidade}", item.categoria) for item in lista_absurda.ingredientes],
       titulo="válida pelo contrato")

válida pelo contrato                            
                                                
  ingrediente          quantidade   categoria   
 ────────────────────────────────────────────── 
  sal                  3 kg         mercearia   
  muçarela de búfala   40 un        laticinios  
  farinha de trigo     250 g        laticinios  
                                                


3 kg de sal para duas pessoas, 40 muçarelas, farinha em laticínios: o enum só confere que cada valor está **na lista**. Decidir se
o valor faz sentido exige **regras** (limites por pessoa, por unidade) e **dados** que o modelo não tem (preço, o que já tem em
casa). Isso é código — funções comuns — e é o assunto do intermediário.

## Fechamento do básico

| Notebook | O que garantiu | O que ficou de fora |
|---|---|---|
| 01 | chamar a API e ler JSON | formato e tipos |
| 02 | forma: campos e tipos (Pydantic) | vocabulário dos campos |
| 03 | vocabulário (enum) e escopo (guardrail) | bom senso dos valores, dados externos |

➡️ [04 — ferramentas com funções](../02_intermediario/04_ferramentas_com_funcoes.ipynb)